## Partie de Paul

In [19]:
import pandas as pd
import numpy as np
import re
from s3_utils import read_s3_csv, get_s3_client, upload_to_s3

In [20]:
s3 = get_s3_client()
BUCKET = "projet-accidents-jedha"

# 1. On liste TOUS les fichiers du dossier bronze
response = s3.list_objects_v2(Bucket=BUCKET, Prefix='bronze/')

# 2. On crée une liste vide pour stocker nos DataFrames
dfs_to_merge = []

# 3. La boucle magique intelligente
for obj in response.get('Contents', []):
    file_key = obj['Key']
    
    # On filtre pour ne garder que la catégorie "caract"
    if "lieux" in file_key.lower():
        
        # --- LOGIQUE D'EXTRACTION DE L'ANNÉE ---
        # On cherche 4 chiffres dans le nom (ex: 2024)
        match = re.search(r'(\d{4})', file_key)
        
        if match:
            annee = int(match.group(1))
            
            # On définit le séparateur selon l'année
            if annee >= 2024:
                sep = ','
            else:
                sep = ';'
            
            print(f"🔎 Trouvé : {file_key} | Année: {annee} | Séparateur: '{sep}'")
            
            # On lit le fichier avec le bon séparateur
            df_temp = read_s3_csv(file_key, separator=sep)
            dfs_to_merge.append(df_temp)

# 4. On fusionne tout d'un coup
if dfs_to_merge:
    lieux_df = pd.concat(dfs_to_merge, ignore_index=True)
    print(f"✅ Terminé ! {len(dfs_to_merge)} fichiers fusionnés.")
else:
    print("⚠️ Aucun fichier trouvé, vérifie le mot-clé ou le dossier.")

🔎 Trouvé : bronze/Dataset - Sécurité Routière.xlsx - lieux 2021.csv | Année: 2021 | Séparateur: ';'
🔎 Trouvé : bronze/Dataset - Sécurité Routière.xlsx - lieux 2022.csv | Année: 2022 | Séparateur: ';'


c:\Users\bsacu\Desktop\FullStack\Projet final\Python\projet-groupe-jedha\s3_utils.py:23: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.BytesIO(response['Body'].read()), sep=separator)


🔎 Trouvé : bronze/Dataset - Sécurité Routière.xlsx - lieux 2023.csv | Année: 2023 | Séparateur: ';'


c:\Users\bsacu\Desktop\FullStack\Projet final\Python\projet-groupe-jedha\s3_utils.py:23: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.BytesIO(response['Body'].read()), sep=separator)


🔎 Trouvé : bronze/Dataset - Sécurité Routière.xlsx - lieux 2024.csv | Année: 2024 | Séparateur: ','
✅ Terminé ! 4 fichiers fusionnés.


In [21]:
# Je le mets en stase au cas ou.

"""df_2021 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - lieux 2021.csv', sep=';')
df_2022 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - lieux 2022.csv', sep=';')
df_2023 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - lieux 2023.csv', sep=';')
df_2024 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - lieux 2024.csv', sep=',')

# Now concatenate
lieux_df = pd.concat([df_2024, df_2023, df_2022, df_2021], ignore_index=True)

print(lieux_df.head())
print(f"\nLe dataset contient :{lieux_df.shape[1]} colonnes\n")"""

'df_2021 = pd.read_csv(\'Dataset/Dataset - Sécurité Routière.xlsx - lieux 2021.csv\', sep=\';\')\ndf_2022 = pd.read_csv(\'Dataset/Dataset - Sécurité Routière.xlsx - lieux 2022.csv\', sep=\';\')\ndf_2023 = pd.read_csv(\'Dataset/Dataset - Sécurité Routière.xlsx - lieux 2023.csv\', sep=\';\')\ndf_2024 = pd.read_csv(\'Dataset/Dataset - Sécurité Routière.xlsx - lieux 2024.csv\', sep=\',\')\n\n# Now concatenate\nlieux_df = pd.concat([df_2024, df_2023, df_2022, df_2021], ignore_index=True)\n\nprint(lieux_df.head())\nprint(f"\nLe dataset contient :{lieux_df.shape[1]} colonnes\n")'

In [22]:
columns_drop = ['voie','v1','v2','circ','vosp','prof','pr','pr1','plan','lartpc','larrout','situ']
lieux_df.drop(columns=columns_drop, inplace= True)

In [23]:
for colonne in lieux_df:
    lieux_df[colonne]=lieux_df[colonne].replace("-1",np.nan)

In [24]:
print(f"\nLe dataset contient :{lieux_df.shape[1]} colonnes\n")
lieux_df.info()


Le dataset contient :6 colonnes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 252928 entries, 0 to 252927
Data columns (total 6 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   Num_Acc  252928 non-null  int64 
 1   catr     252928 non-null  int64 
 2   nbv      248750 non-null  object
 3   surf     252928 non-null  int64 
 4   infra    252928 non-null  int64 
 5   vma      252928 non-null  int64 
dtypes: int64(5), object(1)
memory usage: 11.6+ MB


In [25]:
print("Percentage of missing values: ")
display(100*lieux_df.isnull().sum()/lieux_df.shape[0])

Percentage of missing values: 


Num_Acc    0.000000
catr       0.000000
nbv        1.651853
surf       0.000000
infra      0.000000
vma        0.000000
dtype: float64

In [26]:
print("Percentage of missing values: ")
display(100*lieux_df.notna().sum()/lieux_df.shape[0])

Percentage of missing values: 


Num_Acc    100.000000
catr       100.000000
nbv         98.348147
surf       100.000000
infra      100.000000
vma        100.000000
dtype: float64

In [27]:
"""lieux_df.to_csv('Dataset/lieux.csv', index=False)"""

# On utilise ta fonction d'upload
# 'lieux_cleaned.csv_test' est le nom que le fichier aura sur S3
upload_to_s3(lieux_df, "lieux_cleaned.csv", folder="silver")

✅ Mission accomplie : silver/lieux_cleaned.csv est sur S3 !
